In [3]:
# Optional: set to an integer for a quick smoke test, or None for full run.
TEST_LIMIT = None

MAX_SINGLE_LINE_RETRIES = 0  # Retries don't seem to help, so set to 0 for now.
VERBOSE = True  # Set to True to get more info about retries and single line issues.
LOG_FILE_POSTFIX = "_v1"  # If empty (""), logfile falls back to timestamp mode.

# Focus op (bijvoorbeeld) 3489825760_4 die continu een single line teruggeeft, zelfs na 10 retries. 
# Mogelijk is er iets speciaals aan deze file dat de LLM in de war brengt, of is er een bug in 
# de code die deze file niet goed verwerkt?



In [ ]:
from __future__ import annotations

import json
import logging
import random
import re
import time
from datetime import datetime
from pathlib import Path

from dotenv import dotenv_values
from tqdm.auto import tqdm

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims


def format_seconds(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def to_single_line(claim: str) -> str:
    # Ensure each output line is one atomic string without embedded newlines.
    return " ".join(str(claim).split())


# Last-word patterns that should NOT trigger a sentence split (abbreviations / initials).
_ABBREV_RE = re.compile(
    r'^(?:[A-Z](\.[A-Z])+|[A-Z]|Dr|St|Mr|Mrs|Ms|Prof|Jr|Sr|vs|etc|Inc|Ltd|Co|No|Fig|Dept|Ave|Blvd|Rd|Mt|Lt|Col|Gen|Rep|Gov|Sen|Maj|G\.E)$',
    re.IGNORECASE,
)


def split_claims_on_period(text: str) -> list[str]:
    """Split a single-line string into multiple claims on '. ' boundaries.

    Avoids splitting on common abbreviations (Dr., St., Mr., Prof., etc.) and
    single-letter initials. Each resulting claim is trimmed and ends with '.'.
    Empty parts are discarded.
    """
    raw_parts = text.split(". ")
    merged: list[str] = []
    buffer = raw_parts[0] if raw_parts else ""

    for part in raw_parts[1:]:
        stripped = buffer.rstrip()
        last_word = stripped.rsplit(None, 1)[-1] if stripped else ""
        last_word = last_word.lstrip("\"'(\u2018\u2019\u201c\u201d")
        if _ABBREV_RE.fullmatch(last_word):
            # Last word before '.' is an abbreviation — do not split here.
            buffer = buffer + ". " + part
        else:
            merged.append(buffer)
            buffer = part

    if buffer:
        merged.append(buffer)

    result = []
    for part in merged:
        part = part.strip()
        if not part:
            continue
        if not part.endswith("."):
            part += "."
        result.append(part)
    return result


# Paths and folders
dataset_path = Path("datasets/ContraDoc/ContraDoc.json")
output_dir = Path("extracted_claims")
logs_dir = Path("logs")
output_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# Logging setup (fixed file with postfix, timestamp fallback when postfix is empty)
log_file_postfix = str(globals().get("LOG_FILE_POSTFIX", "_v1")).strip()
if log_file_postfix:
    log_filename = f"extraction_txt{log_file_postfix}.txt"
else:
    run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = f"extraction_txt_{run_ts}.txt"
log_path = logs_dir / log_filename
logger = logging.getLogger("extraction_txt")
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.propagate = False
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(file_handler)

# Model config from .env
env = dotenv_values(DOTENV_PATH)
remote_model_name_raw = env.get("CLAIM_MODEL_REMOTE")
remote_model_name = str(remote_model_name_raw).strip() if remote_model_name_raw else "gpt-4o-mini"

# 1) Load dataset in memory (only id + text)
with dataset_path.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

# Check if dataset contains duplicate IDs
id_counts = {}
for split_name in ("pos", "neg"):
    split_dict = dataset.get(split_name, {})
    if not isinstance(split_dict, dict):
        continue
    for raw in split_dict.values():
        record_id = str(raw.get("unique id", "")).strip()
        if record_id:
            id_counts[record_id] = id_counts.get(record_id, 0) + 1

# Check for duplicate IDs
duplicate_ids = [record_id for record_id, count in id_counts.items() if count > 1]
if duplicate_ids:
    logger.error(f"FOUT: Dubbele IDs gevonden: {duplicate_ids}")
    raise ValueError(f"Dubbele IDs gevonden: {duplicate_ids}")

records = []
for split_name in ("pos", "neg"):
    split_dict = dataset.get(split_name, {})
    if not isinstance(split_dict, dict):
        continue
    for raw in split_dict.values():
        record_id = str(raw.get("unique id", "")).strip()
        text = str(raw.get("text", "")).strip()
        if not record_id:
            logger.error("<ONBEKENDE_ID>: FOUT: ontbrekende unique id")
            continue
        if not text:
            logger.error(f"{record_id}: FOUT: ontbrekende tekst")
            continue
        records.append({"id": record_id, "text": text})

# 2) Remove IDs already present as ID.txt in extracted_claims
existing_ids = {p.stem for p in output_dir.glob("*.txt") if p.is_file()}
pending_records = [r for r in records if r["id"] not in existing_ids]

# 3) Shuffle order in memory
random.shuffle(pending_records)

if TEST_LIMIT is not None:
    pending_records = pending_records[:TEST_LIMIT]

# Run counters and timing
success_count = 0
fail_count = 0
processed_count = 0
total_count = len(pending_records)
run_start = time.perf_counter()

print(f"Loaded records: {len(records)}")
print(f"Already existing .txt files: {len(existing_ids)}")
print(f"Pending records to process: {total_count}")
print(f"Remote model: {remote_model_name}")
print(f"Log file: {log_path}")
print(f"Test limit: {TEST_LIMIT}")

# 4) Extract + write loop with fail-safe on timeout/errors
progress = tqdm(pending_records, total=total_count, desc="Claims extracten", unit="item")
for rec in progress:
    record_id = rec["id"]
    text = rec["text"]
    item_start = time.perf_counter()

    try:
        normalized_lines: list[str] = []
        for attempt in range(1, MAX_SINGLE_LINE_RETRIES + 2):  # attempts 1, 2, 3
            claims = extract_claims(
                text=text,
                model_name=remote_model_name,
                backend="remote",
                use_claimify=False,
                verbose=VERBOSE,
            )
            normalized_lines = [to_single_line(c) for c in claims if str(c).strip()]

            if len(normalized_lines) != 1:
                break  # multiple lines returned — no retry needed

            if attempt <= MAX_SINGLE_LINE_RETRIES:
                logger.warning(
                    f"{record_id}: LLM gaf 1 regel terug (poging {attempt}/{MAX_SINGLE_LINE_RETRIES + 1}), opnieuw proberen..."
                )

        # After all attempts: if still one line, skip and log — do not write.
        if len(normalized_lines) == 1:
            fail_count += 1
            single_line_output = normalized_lines[0]
            logger.error(
                f"{record_id}: OVERGESLAGEN — na {MAX_SINGLE_LINE_RETRIES + 1} pogingen nog steeds 1 regel teruggegeven door LLM."
            )
            logger.error(f"{record_id}: ONE_LINE_OUTPUT: {single_line_output}")
            processed_count += 1
            elapsed = time.perf_counter() - run_start
            avg_per_item = elapsed / processed_count if processed_count else 0.0
            remaining_items = total_count - processed_count
            remaining_seconds = avg_per_item * remaining_items
            total_est_seconds = avg_per_item * total_count
            item_seconds = time.perf_counter() - item_start
            progress.set_postfix(
                goed=success_count,
                fout=fail_count,
                item_s=f"{item_seconds:.2f}",
                elapsed=format_seconds(elapsed),
                remain=format_seconds(remaining_seconds),
                total_est=format_seconds(total_est_seconds),
            )
            continue

        output_path = output_dir / f"{record_id}.txt"
        output_text = "\n".join(normalized_lines)
        output_path.write_text(output_text, encoding="utf-8")

        success_count += 1
        logger.info(f"{record_id}: Ingelezen en weggeschreven naar {output_path}")

    except Exception as exc:
        fail_count += 1
        logger.error(f"{record_id}: FOUT: {exc}")
        # Do not break on timeout or any remote failure; continue with next record.

    processed_count += 1
    elapsed = time.perf_counter() - run_start
    avg_per_item = elapsed / processed_count if processed_count else 0.0
    remaining_items = total_count - processed_count
    remaining_seconds = avg_per_item * remaining_items
    total_est_seconds = avg_per_item * total_count
    item_seconds = time.perf_counter() - item_start

    progress.set_postfix(
        goed=success_count,
        fout=fail_count,
        item_s=f"{item_seconds:.2f}",
        elapsed=format_seconds(elapsed),
        remain=format_seconds(remaining_seconds),
        total_est=format_seconds(total_est_seconds),
    )

run_elapsed = time.perf_counter() - run_start
avg_item = run_elapsed / processed_count if processed_count else 0.0

print("\nRun summary")
print(f"Processed: {processed_count}/{total_count}")
print(f"Goed: {success_count} | Fout: {fail_count}")
print(f"Elapsed: {format_seconds(run_elapsed)}")
print(f"Gem. tijd per item: {avg_item:.2f}s")
print(f"Logbestand: {log_path}")


Loaded records: 891
Already existing .txt files: 782
Pending records to process: 109
Remote model: gpt-4o-mini
Log file: logs/extraction_txt_v1.txt
Test limit: None


Claims extracten:   0%|          | 0/109 [00:00<?, ?item/s]

Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 88 claims.
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
One-line claim output detected after first extraction attempt.
First attempt normalized one-line output: A man named John Lewis is a suspect in the fatal shooting of a Philadelphia police officer. John Lewis was captured at a shelter in Miami, Florida, on Tuesday. Philadelphia Police Commissioner Sylvester Johnson announced the capture at a news conference. Miami Police Chief John Timoney was alerted by Philadelphia police about John Lewis. John Lewis took a Miami-bound bus from Philadelphia. Miami Police Chief John Timoney acted on information from Miami Rescue Mission. Officers went to a homeless shelter to find John Lewis. John Lewis is 6 feet tall and weighs 270 pounds. John Lewis was apprehended unarmed and without incident. It did not appear that anyone in Miami was helping John Lewis. Miami Po

In [ ]:
def correct_single_line_files(extracted_claims_dir: Path, logs_dir: Path) -> None:
    """Read all .txt files in extracted_claims_dir and re-write any single-line
    files by splitting on '. ' boundaries.

    Logs results to logs/correction_<timestamp>.txt:
      - Succesfully corrected files: filename + new line count.
      - Files that could not be corrected: filename + reason.
    """
    correction_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    correction_log_path = logs_dir / f"correction_{correction_ts}.txt"

    corr_logger = logging.getLogger(f"correction_{correction_ts}")
    corr_logger.handlers.clear()
    corr_logger.setLevel(logging.INFO)
    corr_logger.propagate = False
    corr_handler = logging.FileHandler(correction_log_path, encoding="utf-8")
    corr_handler.setFormatter(logging.Formatter("%(message)s"))
    corr_logger.addHandler(corr_handler)

    txt_files = sorted(extracted_claims_dir.glob("*.txt"))
    corrected = 0
    skipped = 0

    for txt_path in txt_files:
        try:
            content = txt_path.read_text(encoding="utf-8")
        except Exception as exc:
            corr_logger.error(f"FOUT  | {txt_path.name} | Kon bestand niet lezen: {exc}")
            skipped += 1
            continue

        lines = [ln for ln in content.splitlines() if ln.strip()]

        # Only process files that consist of exactly one (non-empty) line.
        if len(lines) != 1:
            continue

        single_line = lines[0].strip()
        split_lines = split_claims_on_period(single_line)

        if len(split_lines) <= 1:
            corr_logger.error(
                f"FOUT  | {txt_path.name} | Kon niet corrigeren: geen '. ' scheidingsteken gevonden"
            )
            skipped += 1
            continue

        try:
            txt_path.write_text("\n".join(split_lines), encoding="utf-8")
        except Exception as exc:
            corr_logger.error(f"FOUT  | {txt_path.name} | Kon bestand niet wegschrijven: {exc}")
            skipped += 1
            continue

        corr_logger.info(f"OK    | {txt_path.name} | Gecorrigeerd naar {len(split_lines)} regels")
        corrected += 1

    corr_logger.info(f"\nSamenvatting: {corrected} gecorrigeerd, {skipped} niet gecorrigeerd")
    print(f"Correctie klaar: {corrected} gecorrigeerd, {skipped} niet gecorrigeerd")
    print(f"Logbestand: {correction_log_path}")

# Nu even niet handmatig corrigeren.

# correct_single_line_files(
#     extracted_claims_dir=Path("extracted_claims"),
#     logs_dir=Path("logs"),
# )
